# Job Postings Fact Table Loader

This notebook maintains the `warehouse.fact_job_postings` fact table.

**Purpose**: Core transactional fact for job posting events

**Key Features**:
* One grain: One row per job posting event
* SCD2-aware joins to dimensions using effective date windows
* Foreign keys to all conformed dimensions
* Degenerate dimensions (posting_timestamp)
* Event type flags (is_new_job, is_update, is_soft_delete, is_restore)

**Sources**: 
* `silver.silver_jobs_current` (job events)
* `silver.silver_job_changes` (change detection)
* All dimension tables (FK resolution)

**Target**: `workspace.warehouse.fact_job_postings`

In [0]:
%sql
-- Extract job posting events from silver layer
-- Use most recent change event for each job (or treat as INSERT if no change record exists)
CREATE OR REPLACE TEMP VIEW job_events AS
WITH ranked_changes AS (
  SELECT 
    enterprise_job_id,
    change_type,
    change_timestamp,
    ROW_NUMBER() OVER (PARTITION BY enterprise_job_id ORDER BY change_timestamp DESC) as rn
  FROM workspace.silver.silver_job_changes
)
SELECT 
  j.enterprise_job_id,
  j.source_name,
  COALESCE(j.posted_at, j.created_at, j.updated_at) as posting_timestamp,
  j.is_active as active_flag,
  j.soft_delete_flag,
  j.created_at,
  j.updated_at,
  -- Event type classification from most recent change (or default to INSERT)
  COALESCE(c.change_type, 'INSERT') as change_type,
  CASE WHEN COALESCE(c.change_type, 'INSERT') = 'INSERT' THEN TRUE ELSE FALSE END as is_new_job,
  CASE WHEN COALESCE(c.change_type, 'INSERT') = 'UPDATE' THEN TRUE ELSE FALSE END as is_update,
  CASE WHEN j.soft_delete_flag = TRUE THEN TRUE ELSE FALSE END as is_soft_delete,
  CASE WHEN COALESCE(c.change_type, 'INSERT') = 'RESTORE' THEN TRUE ELSE FALSE END as is_restore
FROM workspace.silver.silver_jobs_current j
LEFT JOIN ranked_changes c
  ON j.enterprise_job_id = c.enterprise_job_id
  AND c.rn = 1
WHERE j.enterprise_job_id IS NOT NULL;

In [0]:
%sql
-- Resolve foreign keys using current dimension records
-- NOTE: For initial load, use current versions since posting timestamps predate dim_job creation
-- In production with ongoing SCD2 tracking, switch to time-travel joins with effective_from/effective_to
CREATE OR REPLACE TEMP VIEW fact_with_fks AS
SELECT 
  e.*,
  -- Job SK - resolve to current version
  COALESCE(j.job_sk, -1) as job_sk,
  -- Company SK - from the job version
  COALESCE(j.company_sk, -1) as company_sk,
  -- Location SK - from the job version
  COALESCE(j.location_sk, -1) as location_sk,
  -- Role SK - from the job version
  COALESCE(dr.role_sk, -1) as role_sk,
  -- Sector SK - from the job version
  COALESCE(j.sector_sk, -1) as sector_sk,
  -- Source SK
  COALESCE(ds.source_sk, -1) as source_sk,
  -- Date SK - convert posting date to YYYYMMDD format
  CAST(DATE_FORMAT(e.posting_timestamp, 'yyyyMMdd') AS INT) as posting_date_sk
FROM job_events e
-- Join to current job dimension version
LEFT JOIN workspace.warehouse.dim_job j
  ON e.enterprise_job_id = j.enterprise_job_id
  AND j.is_current = TRUE
-- Role dimension (NOTE: FK resolution may fail due to dim_job having plain text names
-- instead of structured IDs. This will default role_sk to -1 for unmatched records.)
LEFT JOIN workspace.warehouse.dim_role dr
  ON j.canonical_role_id = dr.canonical_role_id
  AND j.canonical_role_id IS NOT NULL
-- Source dimension
LEFT JOIN workspace.warehouse.dim_source ds
  ON e.source_name = ds.source_name;

In [0]:
%sql
-- Generate fact surrogate keys
CREATE OR REPLACE TEMP VIEW fact_final AS
SELECT
  ROW_NUMBER() OVER (ORDER BY fk.enterprise_job_id, fk.posting_timestamp) + max_keys.max_sk as fact_job_posting_sk,
  fk.job_sk,
  fk.company_sk,
  fk.location_sk,
  fk.role_sk,
  fk.sector_sk,
  fk.source_sk,
  fk.posting_date_sk,
  fk.posting_timestamp,
  fk.active_flag,
  fk.is_new_job,
  fk.is_update,
  fk.is_soft_delete,
  fk.is_restore
FROM fact_with_fks fk
CROSS JOIN (
  SELECT COALESCE(MAX(fact_job_posting_sk), 0) as max_sk 
  FROM workspace.warehouse.fact_job_postings
) max_keys
WHERE fk.job_sk IS NOT NULL AND fk.job_sk != -1;

In [0]:
%sql
-- Merge into fact table (idempotent)
MERGE INTO workspace.warehouse.fact_job_postings target
USING fact_final source
ON target.fact_job_posting_sk = source.fact_job_posting_sk
WHEN MATCHED THEN UPDATE SET
  target.active_flag = source.active_flag,
  target.is_soft_delete = source.is_soft_delete
WHEN NOT MATCHED THEN INSERT (
  fact_job_posting_sk,
  job_sk,
  company_sk,
  location_sk,
  role_sk,
  sector_sk,
  source_sk,
  posting_date_sk,
  posting_timestamp,
  active_flag,
  is_new_job,
  is_update,
  is_soft_delete,
  is_restore
) VALUES (
  source.fact_job_posting_sk,
  source.job_sk,
  source.company_sk,
  source.location_sk,
  source.role_sk,
  source.sector_sk,
  source.source_sk,
  source.posting_date_sk,
  source.posting_timestamp,
  source.active_flag,
  source.is_new_job,
  source.is_update,
  source.is_soft_delete,
  source.is_restore
);

In [0]:
%sql
-- Validate fact table
SELECT 
  COUNT(*) as total_postings,
  COUNT(DISTINCT job_sk) as unique_jobs,
  SUM(CASE WHEN active_flag THEN 1 ELSE 0 END) as active_postings,
  SUM(CASE WHEN is_new_job THEN 1 ELSE 0 END) as new_jobs,
  SUM(CASE WHEN is_update THEN 1 ELSE 0 END) as updated_jobs,
  SUM(CASE WHEN is_soft_delete THEN 1 ELSE 0 END) as deleted_jobs,
  MIN(posting_timestamp) as earliest_posting,
  MAX(posting_timestamp) as latest_posting
FROM workspace.warehouse.fact_job_postings;

-- Sample fact records with dimension context
SELECT 
  f.fact_job_posting_sk,
  j.enterprise_job_id,
  j.title_normalized,
  c.company_name,
  l.location_name,
  r.role_name,
  s.source_name,
  f.posting_timestamp,
  f.active_flag,
  f.is_new_job,
  CASE WHEN f.role_sk = -1 THEN 'Missing Role FK' ELSE 'OK' END as role_status
FROM workspace.warehouse.fact_job_postings f
INNER JOIN workspace.warehouse.dim_job j ON f.job_sk = j.job_sk AND j.is_current = TRUE
INNER JOIN workspace.warehouse.dim_company c ON f.company_sk = c.company_sk
INNER JOIN workspace.warehouse.dim_location l ON f.location_sk = l.location_sk
LEFT JOIN workspace.warehouse.dim_role r ON f.role_sk = r.role_sk AND f.role_sk != -1
INNER JOIN workspace.warehouse.dim_source s ON f.source_sk = s.source_sk
ORDER BY f.posting_timestamp DESC
LIMIT 20;